In [1]:
from pathlib import Path
import json, re

def _read_text(p: Path) -> str:
    return p.read_text(encoding="utf-8", errors="replace")

def _parse_summary_txt(summary_path):
    """
    Parses the summary file written by your script.
    Returns:
      - raw_text
      - kv: key->value for 'key: value' lines
      - per_rank: list of dicts parsed from 'rank ...' lines (timing + affinity)
    """
    summary_path = Path(summary_path)
    txt = _read_text(summary_path)

    kv = {}
    per_rank = []
    # key: value lines (keep simple and robust)
    for line in txt.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            kv[k.strip()] = v.strip()

        # parse the "rank ... @ host: ..." lines your script writes
        m = re.match(
            r"\s*rank\s+(?P<rank>\d+)\s+@\s+(?P<host>[^:]+):\s+"
            r"wall=(?P<wall>[0-9]*\.?[0-9]+)s,\s+cpu=(?P<cpu>[0-9]*\.?[0-9]+)s,\s+rss_peak=(?P<rss>.*)\s*$",
            line,
        )
        if m:
            d = m.groupdict()
            per_rank.append({
                "rank": int(d["rank"]),
                "host": d["host"].strip(),
                "wall_s": float(d["wall"]),
                "cpu_s": float(d["cpu"]),
                "rss_peak_mib": None if d["rss"].strip() in ("None", "") else float(d["rss"]),
            })

    return {"path": str(summary_path), "raw_text": txt, "kv": kv, "per_rank": per_rank}

def _load_manifest_json(manifest_path):
    manifest_path = Path(manifest_path)
    data = json.loads(_read_text(manifest_path))
    return {"path": str(manifest_path), "data": data}

def _infer_run_name_from_files(out_dir):
    """
    If you don't want to type run_name, this tries to find exactly one '*_summary.txt'
    and returns its stem without '_summary'.
    """
    out_dir = Path(out_dir)
    candidates = sorted(out_dir.glob("*_summary.txt"))
    if len(candidates) != 1:
        raise ValueError(f"Expected exactly 1 *_summary.txt in {out_dir}, found {len(candidates)}")
    stem = candidates[0].stem
    if not stem.endswith("_summary"):
        raise ValueError(f"Unexpected summary name: {candidates[0].name}")
    return stem[:-len("_summary")]

def _try_hdf5_metadata(hdf5_path):
    """
    Optional shallow HDF5 inspection: list top-level keys and shapes.
    If h5py isn't available, returns an informative message instead.
    """
    hdf5_path = Path(hdf5_path)
    if not hdf5_path.exists():
        return {"path": str(hdf5_path), "exists": False}

    try:
        import h5py  # type: ignore
    except Exception as e:
        return {"path": str(hdf5_path), "exists": True, "h5py_error": f"{type(e).__name__}: {e}"}

    meta = {"path": str(hdf5_path), "exists": True, "top": {}}
    with h5py.File(hdf5_path, "r") as f:
        for k in f.keys():
            obj = f[k]
            if hasattr(obj, "shape"):
                meta["top"][k] = {"shape": tuple(obj.shape), "dtype": str(obj.dtype)}
            else:
                meta["top"][k] = {"type": "group", "n_children": len(obj.keys())}
    return meta

def read_execution_outputs(out_dir, run_name=None, inspect_hdf5=True):
    """
    Main entry point:
      - out_dir: directory containing run outputs
      - run_name: stem like 'grid_TNG50-4_snap033_nspec2_axis-1_nbins1024_lya_si'
                 If None, auto-detect if exactly one *_summary.txt exists.
      - inspect_hdf5: if True, shallow-read .hdf5 keys/shapes (requires h5py)
    Returns a dict with summary, manifest, optional hdf5_meta, and a small computed performance block.
    """
    out_dir = Path(out_dir)
    if run_name is None:
        run_name = _infer_run_name_from_files(out_dir)

    summary_path  = out_dir / f"{run_name}_summary.txt"
    manifest_path = out_dir / f"{run_name}.hdf5.json"
    hdf5_path     = out_dir / f"{run_name}.hdf5"

    summary = _parse_summary_txt(summary_path)
    manifest = _load_manifest_json(manifest_path)
    hdf5_meta = _try_hdf5_metadata(hdf5_path) if inspect_hdf5 else {"path": str(hdf5_path), "skipped": True}

    # Compact computed performance metrics from summary.txt contents
    kv = summary["kv"]
    def fnum(key):
        try:
            return float(kv.get(key))
        except Exception:
            return None

    wall = fnum("Wall time (max over ranks)")
    cpu  = fnum("CPU time (sum over ranks)")
    ranks = None
    try:
        ranks = int(kv.get("MPI ranks (processes)", "").split()[0])
    except Exception:
        pass

    perf = {
        "mpi_ranks": ranks,
        "wall_max_s": wall,
        "cpu_sum_s": cpu,
        "cpu_over_wall": (cpu / wall) if (cpu is not None and wall not in (None, 0.0)) else None,
    }

    return {
        "out_dir": str(out_dir),
        "run_name": run_name,
        "files": {
            "summary": str(summary_path),
            "manifest": str(manifest_path),
            "hdf5": str(hdf5_path),
        },
        "summary": summary,
        "manifest": manifest,
        "hdf5_meta": hdf5_meta,
        "perf": perf,
    }

def show_execution_summary(run_bundle):
    """
    Pretty, compact display focused on what your execution files store.
    """
    kv = run_bundle["summary"]["kv"]
    perf = run_bundle["perf"]
    man = run_bundle["manifest"]["data"]

    print("\n=== Execution summary (from *_summary.txt + *.hdf5.json) ===")
    print(f"Run name : {run_bundle['run_name']}")
    print(f"Out dir  : {run_bundle['out_dir']}")
    print("\nCore outputs:")
    print("  savepath :", kv.get("savepath"))
    print("  manifest :", kv.get("manifest"))
    print("  tau_keys :", kv.get("tau_keys"))

    print("\nParallelism + placement:")
    print("  MPI ranks (processes):", kv.get("MPI ranks (processes)"))
    print("  Nodes used           :", kv.get("Nodes used"))

    print("\nThread env (per rank):")
    print("  ", kv.get("Thread env (per rank)"))

    print("\nTiming:")
    print("  Wall max [s] :", kv.get("Wall time (max over ranks)"))
    print("  CPU sum  [s] :", kv.get("CPU time (sum over ranks)"))
    if perf["cpu_over_wall"] is not None:
        print(f"  CPU/WALL   : {perf['cpu_over_wall']:.3f}  (CPU-bound ideal ~ ranks)")

    print("\nAffinity + memory (as recorded by the run):")
    # show per-rank lines if present
    pr = run_bundle["summary"]["per_rank"]
    if pr:
        for r in sorted(pr, key=lambda x: x["rank"]):
            print(f"  rank {r['rank']:3d} @ {r['host']}: wall={r['wall_s']:.3f}s cpu={r['cpu_s']:.3f}s rss_peak_mib={r['rss_peak_mib']}")
    else:
        print("  (no per-rank lines parsed)")

    # A couple useful manifest fields (robust: print only if present)
    print("\nManifest highlights:")
    for k in ["savepath", "tau_keys", "lines", "base", "num", "nspec", "axis", "nbins", "preset"]:
        if k in man:
            print(f"  {k}: {man[k]}")
        elif isinstance(man.get("args"), dict) and k in man["args"]:
            print(f"  args.{k}: {man['args'][k]}")

    # Optional HDF5 keys/shapes
    h5m = run_bundle.get("hdf5_meta", {})
    if h5m.get("exists") and "top" in h5m:
        print("\nHDF5 top-level datasets:")
        for k, v in h5m["top"].items():
            print(f"  {k}: {v}")
    elif h5m.get("exists") and h5m.get("h5py_error"):
        print("\nHDF5 inspection:", h5m["h5py_error"])
    print("===========================================================\n")

In [2]:
axis = 1
out_dir = "/home/STORAGE/SKEWERS/TNG50-2/snapdir_033/axis_" + str(axis)
run_name = "grid_TNG50-2_snap033_nspec512_axis"+str(axis)+"_nbins1024_lya_si"

bundle = read_execution_outputs(out_dir, run_name=run_name, inspect_hdf5=True)
show_execution_summary(bundle)


=== Execution summary (from *_summary.txt + *.hdf5.json) ===
Run name : grid_TNG50-2_snap033_nspec512_axis1_nbins1024_lya_si
Out dir  : /home/STORAGE/SKEWERS/TNG50-2/snapdir_033/axis_1

Core outputs:
  savepath : /data/desi/scratch/TNG/SKEWERS/TNG50-2/snapdir_033/axis_1/grid_TNG50-2_snap033_nspec512_axis1_nbins1024_lya_si.hdf5
  manifest : /data/desi/scratch/TNG/SKEWERS/TNG50-2/snapdir_033/axis_1/grid_TNG50-2_snap033_nspec512_axis1_nbins1024_lya_si.hdf5.json
  tau_keys : ['H1_1215', 'Si3_1206', 'Si2_1190', 'Si2_1193']

Parallelism + placement:
  MPI ranks (processes): 24
  Nodes used           : 1  (tdm003.pic.es)

Thread env (per rank):
   OMP=1, MKL=1, OPENBLAS=1

Timing:
  Wall max [s] : 5559.767 s
  CPU sum  [s] : 132435.552 s

Affinity + memory (as recorded by the run):
  rank   0 @ tdm003.pic.es: wall=5559.767s cpu=5513.190s rss_peak_mib=7980.4765625
  rank   1 @ tdm003.pic.es: wall=5559.766s cpu=5516.859s rss_peak_mib=8011.86328125
  rank   2 @ tdm003.pic.es: wall=5559.766s cpu